<a href="https://colab.research.google.com/github/EridalgoRamos/Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG/blob/main/AULA_04/AULA_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# FASE 1: SETUP DE SEGURANÇA E EXTRAÇÃO DE DADOS
# ==============================================================================
!pip install pymupdf4llm langchain-text-splitters sentence-transformers -q

import os
import glob
from google.colab import userdata
import pymupdf4llm

print("🔐 1. Verificando Cofre de Segurança...")
try:
    # Acessando chaves de forma oculta e segura
    EMAIL_GITHUB = userdata.get('EMAIL_GITHUB')
    USUARIO_GITHUB = userdata.get('USUARIO_GITHUB')
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✅ Credenciais validadas no cofre (Secrets).")
except Exception as e:
    print("⚠️ ALERTA: Libere o acesso aos 'Secrets' no menu lateral para este notebook.")

# Configuração de Diretórios exigida pela atividade
PASTA_PDFS = '/content/pdfs_originais'
PASTA_RESULTS = '/content/results'

print("\n📂 2. Mapeando e Estruturando Diretórios...")
arquivos_pdf = glob.glob(f'{PASTA_PDFS}/*.pdf')

if not arquivos_pdf:
    print("⚠️ ERRO: Nenhum PDF encontrado. Crie a pasta 'pdfs_originais' e faça o upload.")
else:
    print(f"✅ Encontrados {len(arquivos_pdf)} documentos. Iniciando extração estruturada...")

    # Processo de Extração de PDF para Markdown
    for pdf_path in arquivos_pdf:
        # Extrai o nome base do documento (ex: 'documento_01')
        nome_base = os.path.splitext(os.path.basename(pdf_path))[0]

        # Cria a árvore de diretórios: results/documento_01/markdown/
        dir_markdown = os.path.join(PASTA_RESULTS, nome_base, 'markdown')
        os.makedirs(dir_markdown, exist_ok=True)

        caminho_md = os.path.join(dir_markdown, f"{nome_base}.md")

        # Se o MD ainda não existir, realiza a conversão
        if not os.path.exists(caminho_md):
            print(f"   📄 Convertendo: {nome_base}.pdf -> Markdown (Preservando tabelas e estrutura)...")
            # pymupdf4llm é excelente para preservar tabelas em formato Markdown e descartar/referenciar imagens
            md_text = pymupdf4llm.to_markdown(pdf_path)

            with open(caminho_md, 'w', encoding='utf-8') as f:
                f.write(md_text)
        else:
            print(f"   ⏭️ {nome_base}.md já existe. Pulando conversão.")

    print("\n✅ Fase 1 Concluída! Todos os documentos possuem uma versão Markdown intermediária nas pastas corretas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.7 MB/s eta 0:00:00
🔐 1. Verificando Cofre de Segurança...
✅ Credenciais validadas no cofre (Secrets).

📂 2. Mapeando e Estruturando Diretórios...
✅ Encontrados 12 documentos. Iniciando extração estruturada...
   📄 Convertendo: scaling_laws_llm.pdf -> Markdown (Preservando tabelas e estrutura)...

=== Document parser messages ===
Using Tesseract for OCR processing.
   📄 Convertendo: bert_pretraining.pdf -> Markdown (Preservando tabelas e estrutura)...

=== Document parser messages ===
Using Tesseract for OCR processing.
   📄 Convertendo: retrieval_augmented_generation.pdf -> Markdown (Preservando tabelas e estrutura)...

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR 

In [2]:
# ==============================================================================
# FASE 2: MOTOR DE CHUNKING (10 ESTRATÉGIAS LANGCHAIN)
# ==============================================================================
import os
import glob
import nltk
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter
)

# Garante o download do tokenizador de sentenças
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

PASTA_RESULTS = '/content/results'

print("🧪 Iniciando Motor de Chunking para todos os documentos...")

# Dicionário mestre para guardar os resultados antes da Fase 3
chunks_mestre = {}

# Mapeia todos os markdowns gerados na Fase 1
arquivos_md = glob.glob(f'{PASTA_RESULTS}/*/markdown/*.md')

for caminho_md in arquivos_md:
    nome_doc = os.path.basename(caminho_md).replace('.md', '')
    print(f"\n📄 Processando fatiamento: {nome_doc}")

    with open(caminho_md, 'r', encoding='utf-8') as f:
        texto = f.read()

    chunks_mestre[nome_doc] = {}

    # ---------------------------------------------------------
    # TESTES 1 A 6: Chunking por Tamanho Extremado e Overlap
    # ---------------------------------------------------------
    estrategias_tamanho = {
        "test_01": {"desc": "Fixo_200_sem_overlap", "splitter": RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)},
        "test_02": {"desc": "Fixo_500_sem_overlap", "splitter": RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)},
        "test_03": {"desc": "Fixo_1000_sem_overlap", "splitter": RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)},
        "test_04": {"desc": "Fixo_2000_sem_overlap", "splitter": RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)},
        "test_05": {"desc": "Fixo_500_overlap_50", "splitter": RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)},
        "test_06": {"desc": "Fixo_500_overlap_200", "splitter": RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)}
    }

    for id_teste, config in estrategias_tamanho.items():
        chunks = config["splitter"].split_text(texto)
        chunks_mestre[nome_doc][id_teste] = {"strategy": config["desc"], "chunks": chunks}

    # ---------------------------------------------------------
    # TESTE 7: Por Parágrafo
    # ---------------------------------------------------------
    # Separação dupla de linha é o padrão universal para parágrafos em Markdown
    splitter_p = CharacterTextSplitter(separator="\n\n", chunk_size=1500, chunk_overlap=0, is_separator_regex=False)
    chunks_p = splitter_p.split_text(texto)
    chunks_mestre[nome_doc]["test_07"] = {"strategy": "Por_paragrafo", "chunks": chunks_p}

    # ---------------------------------------------------------
    # TESTE 8: Por Sentença (Agrupadas em 3)
    # ---------------------------------------------------------
    sentencas = nltk.sent_tokenize(texto)
    chunks_s = []
    for i in range(0, len(sentencas), 3):
        grupo = " ".join(sentencas[i:i+3])
        chunks_s.append(grupo)
    chunks_mestre[nome_doc]["test_08"] = {"strategy": "Sentencas_agrupadas_3", "chunks": chunks_s}

    # ---------------------------------------------------------
    # TESTE 9: Recursive Chunking (Separadores Hierárquicos)
    # ---------------------------------------------------------
    # Prioriza manter blocos inteiros; se falhar, tenta sentenças; se falhar, palavras; se falhar, caracteres.
    splitter_r = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", ".", " ", ""], chunk_size=1000, chunk_overlap=100)
    chunks_r = splitter_r.split_text(texto)
    chunks_mestre[nome_doc]["test_09"] = {"strategy": "Recursivo_Hierarquico", "chunks": chunks_r}

    # ---------------------------------------------------------
    # TESTE 10: Markdown / Estrutura Semântica
    # ---------------------------------------------------------
    headers_to_split_on = [("#", "H1"), ("##", "H2"), ("###", "H3"), ("####", "H4")]
    splitter_md = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    # O Markdown splitter retorna objetos 'Document' que já possuem o metadata acoplado (ex: nível do heading)
    chunks_md_raw = splitter_md.split_text(texto)

    # Convertendo para um formato fácil de iterar na Fase 3, preservando os metadados
    chunks_md_formatados = [{"text": doc.page_content, "metadata": doc.metadata} for doc in chunks_md_raw]
    chunks_mestre[nome_doc]["test_10"] = {"strategy": "Markdown_Semantico", "chunks_estruturados": chunks_md_formatados}

print("\n✅ Fase 2 Concluída! Todas as 10 estratégias foram aplicadas e estruturadas em memória.")

🧪 Iniciando Motor de Chunking para todos os documentos...

📄 Processando fatiamento: attention_is_all_you_need

📄 Processando fatiamento: llama_foundation_models

📄 Processando fatiamento: escrita_academica_ia



📄 Processando fatiamento: instruct_gpt



📄 Processando fatiamento: gpt4_technical_report



📄 Processando fatiamento: lora_low_rank_adaptation

📄 Processando fatiamento: retrieval_augmented_generation

📄 Processando fatiamento: twitter_algoritmo



📄 Processando fatiamento: scaling_laws_llm

📄 Processando fatiamento: gpt3_language_models



📄 Processando fatiamento: bert_pretraining

📄 Processando fatiamento: bioetica_e_ia

✅ Fase 2 Concluída! Todas as 10 estratégias foram aplicadas e estruturadas em memória.


In [3]:
# ==============================================================================
# FASE 3: MOTOR DE EMBEDDINGS E EXPORTAÇÃO JSON
# ==============================================================================
import os
import json
from sentence_transformers import SentenceTransformer

# 1. Configuração exigida na atividade
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
print(f"🔄 Carregando modelo de embeddings: {EMBEDDING_MODEL} (Acelerado por GPU T4)...")
modelo = SentenceTransformer(EMBEDDING_MODEL)

PASTA_RESULTS = '/content/results'

# Tabela de configurações para preencher o JSON conforme a atividade
configs_testes = {
    1: {"chunk_size": 200, "chunk_overlap": 0},
    2: {"chunk_size": 500, "chunk_overlap": 0},
    3: {"chunk_size": 1000, "chunk_overlap": 0},
    4: {"chunk_size": 2000, "chunk_overlap": 0},
    5: {"chunk_size": 500, "chunk_overlap": 50},
    6: {"chunk_size": 500, "chunk_overlap": 200},
    7: {"chunk_size": "paragrafo", "chunk_overlap": 0},
    8: {"chunk_size": "3_sentencas", "chunk_overlap": 0},
    9: {"chunk_size": 1000, "chunk_overlap": 100},
    10: {"chunk_size": "secao_markdown", "chunk_overlap": 0}
}

print("\n🧠 Iniciando a geração de Embeddings e estruturação dos JSONs...")

# Iterando sobre os dados gerados na Fase 2
for nome_doc, testes in chunks_mestre.items():
    print(f"\n📄 Vetorizando: {nome_doc}")

    for id_teste, dados_teste in testes.items():
        numero_teste = int(id_teste.split('_')[1])
        nome_estrategia = dados_teste['strategy']
        config = configs_testes[numero_teste]

        # Cria a pasta do teste (ex: results/documento_01/test_01)
        dir_teste = os.path.join(PASTA_RESULTS, nome_doc, id_teste)
        os.makedirs(dir_teste, exist_ok=True)

        # Padroniza a leitura dos chunks
        if id_teste == 'test_10':
            lista_chunks = dados_teste['chunks_estruturados']
        else:
            lista_chunks = [{"text": c if isinstance(c, str) else c.page_content, "metadata": {}} for c in dados_teste['chunks']]

        # Extrai apenas os textos para passar na GPU (processamento em lote para ser ultra rápido)
        textos_para_embed = [c['text'] for c in lista_chunks]

        if textos_para_embed:
            # A mágica acontece aqui: conversão de texto para vetores matemáticos
            embeddings = modelo.encode(textos_para_embed, convert_to_numpy=True).tolist()
        else:
            embeddings = []

        # Montagem do JSON no formato estrito exigido pela atividade
        json_data = []
        for idx, (chunk_dict, embed) in enumerate(zip(lista_chunks, embeddings)):
            obj = {
                "chunk_id": f"{nome_doc}_{id_teste}_chunk{idx+1:03d}",
                "document_id": nome_doc,
                "document_name": f"{nome_doc}.pdf",
                "test_id": numero_teste,
                "strategy": nome_estrategia,
                "chunk_size": config["chunk_size"],
                "chunk_overlap": config["chunk_overlap"],
                "text": chunk_dict['text'],
                "embedding": embed,
                "metadata": chunk_dict.get('metadata', {})
            }
            json_data.append(obj)

        # Salva o arquivo JSON na pasta correta
        caminho_json = os.path.join(dir_teste, 'chunks_embeddings.json')
        with open(caminho_json, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)

        print(f"   ✓ {id_teste}: {len(json_data)} chunks convertidos em matrizes e salvos em JSON.")

print("\n✅ Fase 3 Concluída! Todo o pipeline de Inteligência Artificial foi executado.")

🔄 Carregando modelo de embeddings: all-MiniLM-L6-v2 (Acelerado por GPU T4)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


🧠 Iniciando a geração de Embeddings e estruturação dos JSONs...

📄 Vetorizando: attention_is_all_you_need
   ✓ test_01: 294 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_02: 120 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_03: 52 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_04: 24 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_05: 120 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_06: 132 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_07: 34 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_08: 119 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_09: 53 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_10: 26 chunks convertidos em matrizes e salvos em JSON.

📄 Vetorizando: llama_foundation_models
   ✓ test_01: 626 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_02: 260 chunks convertidos em matrizes e salvos em JSON.
   ✓ test_03: 119 chunks convertidos em matrizes e sal

In [4]:
# ==============================================================================
# FASE 4: GERAÇÃO DO SUMMARY.JSON E ESTATÍSTICAS
# ==============================================================================
import os
import json
import glob
import numpy as np

PASTA_RESULTS = '/content/results'
# Identifica todas as pastas de documentos criadas
documentos = [d for d in os.listdir(PASTA_RESULTS) if os.path.isdir(os.path.join(PASTA_RESULTS, d))]

print("📊 Iniciando a geração dos relatórios summary.json...")

for nome_doc in documentos:
    caminho_doc = os.path.join(PASTA_RESULTS, nome_doc)

    # Estrutura base do JSON exigida na atividade
    summary_data = {
        "document": f"{nome_doc}.pdf",
        "experiments": []
    }

    # Percorre as 10 pastas de testes (test_01 a test_10) dentro do documento
    pastas_testes = sorted(glob.glob(os.path.join(caminho_doc, 'test_*')))

    for pasta_teste in pastas_testes:
        caminho_json = os.path.join(pasta_teste, 'chunks_embeddings.json')

        if os.path.exists(caminho_json):
            with open(caminho_json, 'r', encoding='utf-8') as f:
                dados_chunks = json.load(f)

            num_chunks = len(dados_chunks)

            if num_chunks > 0:
                # Coleta informações do primeiro chunk para servir de base
                primeiro = dados_chunks[0]
                test_id = primeiro['test_id']
                strategy = primeiro['strategy']
                chunk_size = primeiro['chunk_size']
                chunk_overlap = primeiro['chunk_overlap']
                embedding_dim = len(primeiro['embedding'])

                # Calcula o tamanho médio dos chunks (em caracteres)
                tamanhos = [len(c['text']) for c in dados_chunks]
                avg_size = round(np.mean(tamanhos), 2)
            else:
                # Fallback de segurança caso o documento seja muito curto para a estratégia
                test_id = int(os.path.basename(pasta_teste).split('_')[1])
                strategy = "Vazio"
                chunk_size = 0
                chunk_overlap = 0
                embedding_dim = 0
                avg_size = 0.0

            # Monta o bloco do experimento
            experimento = {
                "test_id": test_id,
                "strategy": strategy,
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "num_chunks": num_chunks,
                "avg_chunk_size": avg_size,
                "embedding_dimension": embedding_dim
            }
            summary_data["experiments"].append(experimento)

    # Salva o summary.json na raiz da pasta do documento
    caminho_summary = os.path.join(caminho_doc, 'summary.json')
    with open(caminho_summary, 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, ensure_ascii=False, indent=2)

    print(f"   ✓ summary.json criado com sucesso para: {nome_doc}")

print("\n✅ Fase 4 Concluída! Base de dados comparativa finalizada e pronta para análise.")

📊 Iniciando a geração dos relatórios summary.json...
   ✓ summary.json criado com sucesso para: attention_is_all_you_need
   ✓ summary.json criado com sucesso para: llama_foundation_models
   ✓ summary.json criado com sucesso para: escrita_academica_ia
   ✓ summary.json criado com sucesso para: instruct_gpt
   ✓ summary.json criado com sucesso para: gpt4_technical_report
   ✓ summary.json criado com sucesso para: lora_low_rank_adaptation
   ✓ summary.json criado com sucesso para: retrieval_augmented_generation
   ✓ summary.json criado com sucesso para: twitter_algoritmo
   ✓ summary.json criado com sucesso para: scaling_laws_llm
   ✓ summary.json criado com sucesso para: gpt3_language_models
   ✓ summary.json criado com sucesso para: bert_pretraining
   ✓ summary.json criado com sucesso para: bioetica_e_ia

✅ Fase 4 Concluída! Base de dados comparativa finalizada e pronta para análise.


In [5]:
# ==============================================================================
# FASE 5: ENVIO SEGURO PARA O GITHUB (DADOS E ESTRUTURA)
# ==============================================================================
import os
from google.colab import userdata

print("🔐 1. Acessando o cofre de segurança...")
EMAIL_GITHUB = userdata.get('EMAIL_GITHUB')
USUARIO_GITHUB = userdata.get('USUARIO_GITHUB')
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
NOME_REPO = "Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG"

print("📥 2. Baixando o seu repositório atual...")
%cd /content
# Clonagem silenciosa para não imprimir o token na tela
os.system(f"git clone https://{GITHUB_TOKEN}@github.com/{USUARIO_GITHUB}/{NOME_REPO}.git > /dev/null 2>&1")

print("📂 3. Organizando os arquivos da AULA_04...")
%cd /content/{NOME_REPO}
# Cria a pasta da aula e copia todos os resultados gerados nas fases anteriores
!mkdir -p AULA_04/dados
!cp -r /content/results/* AULA_04/dados/

print("🔐 4. Assinando as credenciais de commit...")
!git config --global user.email "{EMAIL_GITHUB}"
!git config --global user.name "{USUARIO_GITHUB}"

print("🚀 5. Empacotando e enviando para a nuvem de forma oculta...")
!git add AULA_04/
!git commit -m "aula 4 - Pipeline RAG, json e embeddings"

# Envio silencioso rigoroso
comando_push = f"git push https://{GITHUB_TOKEN}@github.com/{USUARIO_GITHUB}/{NOME_REPO}.git main > /dev/null 2>&1"
resultado = os.system(comando_push)

if resultado == 0:
    print("\n✅ Sucesso Absoluto! Toda a sua base de dados estruturada já está segura no GitHub.")
else:
    print("\n⚠️ Ocorreu um erro. Pode ser um limite de tamanho do GitHub ou falha de conexão.")

🔐 1. Acessando o cofre de segurança...
📥 2. Baixando o seu repositório atual...
/content
📂 3. Organizando os arquivos da AULA_04...
/content/Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG
🔐 4. Assinando as credenciais de commit...
🚀 5. Empacotando e enviando para a nuvem de forma oculta...
[main 2eebb5c] aula 4 - Pipeline RAG, json e embeddings
 144 files changed, 12461497 insertions(+)
 create mode 100644 AULA_04/dados/attention_is_all_you_need/markdown/attention_is_all_you_need.md
 create mode 100644 AULA_04/dados/attention_is_all_you_need/summary.json
 create mode 100644 AULA_04/dados/attention_is_all_you_need/test_01/chunks_embeddings.json
 create mode 100644 AULA_04/dados/attention_is_all_you_need/test_02/chunks_embeddings.json
 create mode 100644 AULA_04/dados/attention_is_all_you_need/test_03/chunks_embeddings.json
 create mode 100644 AULA_04/dados/attention_is_all_you_need/test_04/chunks_embeddings.json
 create mode 100644 AULA_04/dados/attention_is_all_you_need/test_05/ch

In [6]:
import os
from google.colab import userdata

print("📝 1. Gerando o arquivo Markdown do Relatório...")

# O texto completo do seu relatório estruturado em Markdown
conteudo_relatorio = """# Relatório Analítico: Avaliação de Estratégias de Chunking com LangChain

**1. Qual estratégia gerou mais chunks?**
A Estratégia 1 (Fixo, 200 caracteres, sem overlap). Por ter um limite de corte extremamente baixo e sem sobreposição, ela estilhaçou os documentos no maior número de fragmentos possível.

**2. Qual gerou menos chunks?**
Geralmente, a Estratégia 4 (Fixo, 2000 caracteres, sem overlap) ou a Estratégia 10 (Markdown Semântico). A estratégia de 2000 caracteres agrupa grandes volumes de texto, enquanto a Markdown cria blocos baseados apenas na ocorrência de subtítulos (que podem ser esparsos em artigos extensos).

**3. Como o tamanho dos chunks variou?**
Nas estratégias 1 a 6 (Fixo), o tamanho se manteve rígido e artificial. Nas estratégias 7 (Parágrafo) e 8 (Sentença), o tamanho variou organicamente conforme a pontuação natural do autor. Na estratégia 10 (Markdown), a variação foi extrema, dependendo do tamanho de cada seção do artigo.

**4. Qual estratégia preservou melhor a estrutura dos documentos?**
A Estratégia 10 (Markdown). Ao fatiar o texto utilizando os *headings* (H1, H2, H3) como delimitadores, ela garantiu que introduções, metodologias e conclusões não fossem misturadas, preservando metadados fundamentais.

**5. Como tabelas foram tratadas?**
Graças à utilização da biblioteca baseada em LLM na conversão, as tabelas foram preservadas em sua estrutura visual de colunas utilizando a formatação nativa do Markdown (ex: `| Coluna A | Coluna B |`). Isso manteve a relação semântica entre os dados tabulares intacta.

**6. Como imagens foram tratadas?**
O processo de OCR identificou a presença das imagens. Na conversão para texto puro/Markdown, a representação visual da imagem é descartada, mas legendas e textos contidos dentro das imagens foram extraídos e convertidos para representação textual.

**7. Quais informações foram perdidas durante a conversão PDF -> Markdown?**
Perdeu-se a formatação de layout de página complexa (como múltiplas colunas visuais), cabeçalhos e rodapés repetitivos que se misturam ao texto principal, e o significado puramente gráfico de diagramas complexos que não puderam ser convertidos em tabelas.

**8. O chunking por caracteres fragmentou conceitos ou estruturas importantes?**
Sim. Nas estratégias sem *overlap* (1 a 4), cortes artificiais muitas vezes ocorreram no meio de uma palavra, de uma tabela ou de um parágrafo crucial, destruindo o contexto exato necessário para o modelo de linguagem compreender a ideia.

**9. O chunking por parágrafo produziu chunks muito grandes?**
Sim. Como documentado nos alertas de execução (*warnings* de tamanho), textos acadêmicos frequentemente contêm blocos densos ou tabelas contínuas que não possuem quebra dupla de linha, forçando o *splitter* a gerar *chunks* maiores que o limite ideal estipulado para não quebrar a estrutura.

**10. O chunking por sentença conseguiu preservar melhor o contexto?**
Preservou o micro-contexto (as frases faziam sentido isoladamente), mas pecou no macro-contexto. Agrupar 3 sentenças de forma rígida pode quebrar um argumento que leva 5 sentenças para ser concluído pelo autor.

**11. O Recursive Splitter apresentou vantagens?**
Sim. Ao utilizar uma lista hierárquica de separadores (tentando primeiro parágrafos, depois sentenças, e só por último caracteres arbitrários), ele funcionou como a estratégia mais inteligente e balanceada para textos não-estruturados, minimizando cortes agressivos.

**12. O Markdown Splitter conseguiu preservar a estrutura semântica?**
De forma excelente. Ele não apenas manteve os blocos de seções unidos, como também embutiu metadados valiosos (informando a qual subtítulo aquele pedaço de texto pertence), o que enriquece enormemente os filtros em um banco de dados vetorial.

**13. Qual estratégia parece mais adequada para um sistema de RAG?**
A Estratégia 9 (Recursive Chunking com overlap) e a Estratégia 10 (Markdown Semântico). A estratégia recursiva é o padrão-ouro para consistência de tamanho dos vetores, enquanto a Markdown é imbatível para garantir respostas coerentes sobre tópicos específicos de um artigo.

**14. Quais estratégias devem ser descartadas?**
As Estratégias 1, 2, 3 e 4 (Fixo sem overlap). A ausência de sobreposição gera perda de contexto nas extremidades de cada *chunk*, prejudicando gravemente a capacidade da IA de recuperar informações na fronteira dos cortes.

**15. Quais estratégias você acha que devem ser utilizadas nos próximos experimentos?**
Uma abordagem híbrida: iniciar com o **Markdown Splitter** (Estratégia 10) para isolar as seções semânticas de forma macro e, em seguida, aplicar o **Recursive Character Text Splitter** (Estratégia 9) dentro de seções que ficaram muito grandes, garantindo *chunks* ricos em contexto e amigáveis ao limite de *tokens* do LLM.
"""

# Caminho do repositório
NOME_REPO = "Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG"
caminho_arquivo = f'/content/{NOME_REPO}/AULA_04/relatorio_analise.md'

# Escreve o arquivo no disco do Colab
with open(caminho_arquivo, 'w', encoding='utf-8') as f:
    f.write(conteudo_relatorio)
print("✅ Arquivo relatorio_analise.md salvo na pasta AULA_04!")

print("\n🔐 2. Autenticando e enviando para o GitHub...")
EMAIL_GITHUB = userdata.get('EMAIL_GITHUB')
USUARIO_GITHUB = userdata.get('USUARIO_GITHUB')
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

%cd /content/{NOME_REPO}

# Comandos Git para adicionar e enviar o relatório
!git config --global user.email "{EMAIL_GITHUB}"
!git config --global user.name "{USUARIO_GITHUB}"
!git add AULA_04/relatorio_analise.md
!git commit -m "aula 4 - Relatório analítico adicionado"

comando_push = f"git push https://{GITHUB_TOKEN}@github.com/{USUARIO_GITHUB}/{NOME_REPO}.git main > /dev/null 2>&1"
resultado = os.system(comando_push)

if resultado == 0:
    print("\n🚀 Sucesso Total! O seu relatório em Markdown já está publicado no seu repositório.")
else:
    print("\n⚠️ Ocorreu um erro no envio. Verifique a conexão.")

📝 1. Gerando o arquivo Markdown do Relatório...
✅ Arquivo relatorio_analise.md salvo na pasta AULA_04!

🔐 2. Autenticando e enviando para o GitHub...
/content/Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG
[main e5a3b02] aula 4 - Relatório analítico adicionado
 1 file changed, 46 insertions(+)
 create mode 100644 AULA_04/relatorio_analise.md

🚀 Sucesso Total! O seu relatório em Markdown já está publicado no seu repositório.
